In [33]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    FunctionTransformer,
    LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

# 1. LOAD DATA

CSV_PATH = "data.csv"

df = pd.read_csv(CSV_PATH)

print("=" * 70)
print("DATASET PROFILE")
print("=" * 70)

print("Shape:", df.shape)
print("\nHead:")
print(df.head())

print("\nTail:")
print(df.tail())

print("\nData types:")
print(df.dtypes)

print("\nInfo:")
df.info()

print("\nNumerical description:")
print(df.describe())

print("\nCategorical description:")
print(df.describe(include="object"))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# 2. UNIQUE VALUES

print("\nUnique values:")
for column in df.columns:
    print(f"{column}: {df[column].nunique()}")


# 3. TARGET ANALYSIS

print("\nTarget distribution:")
print(df["offer_given"].value_counts())

print("\nTarget distribution (%):")
print(df["offer_given"].value_counts(normalize=True) * 100)

# 4. DATA QUALITY / LOGICAL CHECKS
print("\nLogical checks:")

print(
    "Age valid:",
    df["customer_age"].between(18, 100).all()
)

print(
    "Tenure non-negative:",
    (df["customer_tenure_years"] >= 0).all()
)

print(
    "Payment rate valid:",
    df["on_time_payment_rate"].between(0, 100).all()
)

claim_cols = [
    "num_claims_last_year",
    "total_claim_amount_last_year",
    "rejected_claims",
    "claims_approved"
]

print(
    "Claims non-negative:",
    (df[claim_cols] >= 0).all().all()
)

print(
    "Late-payment flag consistent:",
    (
        df["has_late_payments"]
        == (df["late_payment_count"] > 0)
    ).all()
)

print(
    "Premium increase calculation consistent:",
    np.isclose(
        df["premium_increase_pct"],
        (
            (df["premium_amount"]
             - df["previous_premium_amount"])
            / df["previous_premium_amount"]
            * 100
        ),
        atol=0.02
    ).all()
)


# ============================================================
# 5. REDUNDANCY ANALYSIS
# ============================================================

print("\nRedundancy checks:")

print(
    "policy_name == policy_type:",
    (df["policy_name"] == df["policy_type"]).all()
)

print(
    "has_late_payments == late_payment_count > 0:",
    (
        df["has_late_payments"]
        == (df["late_payment_count"] > 0)
    ).all()
)


# ============================================================
# 6. DEFINE FINAL FEATURES
# ============================================================

# Removed:
# customer_id       -> identifier
# policy_id         -> redundant identifier
# policy_type       -> identical to policy_name
# has_late_payments -> exactly derived from late_payment_count

features = [
    "customer_age",
    "customer_gender",
    "customer_occupation",
    "customer_tenure_years",
    "policy_name",
    "premium_amount",
    "previous_premium_amount",
    "premium_increase_pct",
    "payment_frequency",
    "days_to_renewal",
    "late_payment_count",
    "avg_days_late",
    "on_time_payment_rate",
    "num_claims_last_year",
    "total_claim_amount_last_year",
    "rejected_claims",
    "claims_approved"
]

target = "offer_given"

X = df[features].copy()
y = df[target].copy()


# ============================================================
# 7. ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("\nTarget classes:")
for index, class_name in enumerate(label_encoder.classes_):
    print(index, "=", class_name)


# ============================================================
# 8. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain shape before preprocessing:")
print(X_train.shape)

print("\nTest shape before preprocessing:")
print(X_test.shape)

print("\nTraining class counts:")
print(
    pd.Series(
        y_train,
        name="class"
    ).value_counts().sort_index()
)

print("\nTesting class counts:")
print(
    pd.Series(
        y_test,
        name="class"
    ).value_counts().sort_index()
)


# ============================================================
# 9. DEFINE COLUMN GROUPS
# ============================================================

numeric_features = [
    "customer_age",
    "customer_tenure_years",
    "premium_amount",
    "previous_premium_amount",
    "premium_increase_pct",
    "days_to_renewal",
    "late_payment_count",
    "avg_days_late",
    "on_time_payment_rate",
    "num_claims_last_year",
    "total_claim_amount_last_year",
    "rejected_claims",
    "claims_approved"
]

# Highly right-skewed monetary features
log_numeric_features = [
    "premium_amount",
    "previous_premium_amount",
    "total_claim_amount_last_year"
]

# Remaining numerical features
standard_numeric_features = [
    feature
    for feature in numeric_features
    if feature not in log_numeric_features
]

categorical_features = [
    "customer_gender",
    "customer_occupation",
    "policy_name",
    "payment_frequency"
]


# ============================================================
# 10. NUMERICAL PIPELINES
# ============================================================

# Highly skewed numerical variables:
# median imputation -> log1p -> standardization

log_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "log1p",
            FunctionTransformer(
                np.log1p,
                feature_names_out="one-to-one"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# Normally distributed / less severely skewed variables:
# median imputation -> standardization

standard_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# 11. CATEGORICAL PIPELINE
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)



preprocessor = ColumnTransformer(
    transformers=[
        (
            "log_numeric",
            log_numeric_pipeline,
            log_numeric_features
        ),
        (
            "standard_numeric",
            standard_numeric_pipeline,
            standard_numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


# ============================================================
# 13. FIT ONLY ON TRAINING DATA
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)

# IMPORTANT:
# No fitting occurs here.
# The transformations learned from X_train
# are simply applied to X_test.

X_test_processed = preprocessor.transform(X_test)


# ============================================================
# 14. CONVERT TO NUMPY FLOAT32
# ============================================================

X_train_processed = np.asarray(
    X_train_processed,
    dtype=np.float32
)

X_test_processed = np.asarray(
    X_test_processed,
    dtype=np.float32
)


# ============================================================
# 15. VERIFY PROCESSED DATA
# ============================================================

print("\nProcessed X_train shape:")
print(X_train_processed.shape)

print("\nProcessed X_test shape:")
print(X_test_processed.shape)

print("\nNaN in X_train:")
print(np.isnan(X_train_processed).sum())

print("\nInf in X_train:")
print(np.isinf(X_train_processed).sum())

print("\nNaN in X_test:")
print(np.isnan(X_test_processed).sum())

print("\nInf in X_test:")
print(np.isinf(X_test_processed).sum())


# ============================================================
# 16. FINAL FEATURE COUNT
# ============================================================

input_size = X_train_processed.shape[1]

num_classes = len(
    np.unique(y_train)
)

print("\nFinal input size:", input_size)
print("Number of classes:", num_classes)


# ============================================================
# 17. CLASS WEIGHTS
# ============================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("\nClass weights:")
for i, weight in enumerate(class_weights):
    print(
        f"{i} ({label_encoder.classes_[i]}): "
        f"{weight.item():.6f}"
    )


# ============================================================
# 18. CONVERT TO PYTORCH TENSORS
# ============================================================

X_train_tensor = torch.tensor(
    X_train_processed,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_processed,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)


# ============================================================
# 19. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL PYTORCH DATA")
print("=" * 70)

print("X_train_tensor:")
print("Shape:", X_train_tensor.shape)
print("Dtype:", X_train_tensor.dtype)

print("\ny_train_tensor:")
print("Shape:", y_train_tensor.shape)
print("Dtype:", y_train_tensor.dtype)

print("\nX_test_tensor:")
print("Shape:", X_test_tensor.shape)
print("Dtype:", X_test_tensor.dtype)

print("\ny_test_tensor:")
print("Shape:", y_test_tensor.shape)
print("Dtype:", y_test_tensor.dtype)

print("\nInput size:", input_size)
print("Number of classes:", num_classes)

print("\nClass mapping:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{i} -> {class_name}")

print("\nPreprocessing completed successfully.")
print("ANN training has NOT been performed.")

DATASET PROFILE
Shape: (20000, 22)

Head:
  policy_id policy_name customer_id  customer_age customer_gender  \
0      P001   Term Life  CUST000001            33          Female   
1      P001   Term Life  CUST000002            57            Male   
2      P001   Term Life  CUST000003            32          Female   
3      P001   Term Life  CUST000004            34            Male   
4      P001   Term Life  CUST000005            54          Female   

  customer_occupation  customer_tenure_years policy_type  premium_amount  \
0     Sales Executive                    6.2   Term Life            8400   
1   Software Engineer                    3.3   Term Life           16650   
2   Software Engineer                    1.7   Term Life            9750   
3             Teacher                    3.8   Term Life           10050   
4              Farmer                    3.8   Term Life           11950   

   previous_premium_amount  ...  days_to_renewal has_late_payments  \
0               

In [34]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [35]:
# Reproducibility
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [36]:
# Create datasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [37]:
from torch import nn

class InsuranceOfferANN(nn.Module):

    def __init__(self, input_size, num_classes):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.30),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.20),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [38]:
input_size = X_train_tensor.shape[1]
num_classes = 3

model = InsuranceOfferANN(
    input_size=input_size,
    num_classes=num_classes
).to(device)

print(model)

InsuranceOfferANN(
  (network): Sequential(
    (0): Linear(in_features=273, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): ReLU()
    (10): Linear(in_features=32, out_features=3, bias=True)
  )
)


In [39]:
class_weights = torch.tensor(
    [2.321869, 1.159168, 0.585952],
    dtype=torch.float32
).to(device)

In [40]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [41]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [42]:
num_epochs = 50

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        outputs = model(X_batch)

        # Calculate loss
        loss = criterion(outputs, y_batch)

        # Clear gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * X_batch.size(0)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.4f}"
    )

Epoch [1/50] Loss: 0.3421 Accuracy: 0.8079
Epoch [2/50] Loss: 0.1683 Accuracy: 0.9224
Epoch [3/50] Loss: 0.1417 Accuracy: 0.9377
Epoch [4/50] Loss: 0.1235 Accuracy: 0.9441
Epoch [5/50] Loss: 0.1125 Accuracy: 0.9496
Epoch [6/50] Loss: 0.1037 Accuracy: 0.9549
Epoch [7/50] Loss: 0.0886 Accuracy: 0.9627
Epoch [8/50] Loss: 0.0794 Accuracy: 0.9670
Epoch [9/50] Loss: 0.0785 Accuracy: 0.9652
Epoch [10/50] Loss: 0.0653 Accuracy: 0.9736
Epoch [11/50] Loss: 0.0685 Accuracy: 0.9707
Epoch [12/50] Loss: 0.0604 Accuracy: 0.9737
Epoch [13/50] Loss: 0.0590 Accuracy: 0.9756
Epoch [14/50] Loss: 0.0580 Accuracy: 0.9769
Epoch [15/50] Loss: 0.0580 Accuracy: 0.9762
Epoch [16/50] Loss: 0.0532 Accuracy: 0.9771
Epoch [17/50] Loss: 0.0547 Accuracy: 0.9770
Epoch [18/50] Loss: 0.0469 Accuracy: 0.9816
Epoch [19/50] Loss: 0.0452 Accuracy: 0.9822
Epoch [20/50] Loss: 0.0464 Accuracy: 0.9826
Epoch [21/50] Loss: 0.0443 Accuracy: 0.9838
Epoch [22/50] Loss: 0.0396 Accuracy: 0.9838
Epoch [23/50] Loss: 0.0466 Accuracy: 0.98

In [43]:
model.eval()

test_loss = 0.0
correct = 0
total = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        test_loss += (
            loss.item() * X_batch.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            y_batch.cpu().numpy()
        )


test_loss /= total
test_accuracy = correct / total

print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


Test Loss: 0.11299048228934407
Test Accuracy: 0.97


In [44]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print(
    classification_report(
        all_targets,
        all_predictions,
        target_names=[
            "Installment Payment",
            "Loyalty Discount",
            "No-Claim Protection Benefit"
        ]
    )
)

                             precision    recall  f1-score   support

        Installment Payment       0.92      0.95      0.93       574
           Loyalty Discount       0.96      0.98      0.97      1151
No-Claim Protection Benefit       0.99      0.97      0.98      2275

                   accuracy                           0.97      4000
                  macro avg       0.96      0.97      0.96      4000
               weighted avg       0.97      0.97      0.97      4000



In [45]:
cm = confusion_matrix(
    all_targets,
    all_predictions
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 545   14   15]
 [  11 1125   15]
 [  38   27 2210]]


In [46]:
torch.save(model.state_dict(), "insurance_offer_ann.pth")

In [47]:
# Inspect the model output for one batch
with torch.no_grad():
    X_batch_sample, y_batch_sample = next(iter(test_loader))
    X_batch_sample = X_batch_sample.to(device)
    logits = model(X_batch_sample)
    probs = torch.softmax(logits, dim=1)
    pred_idx = torch.argmax(logits, dim=1)

print("Sample input shape:", tuple(X_batch_sample.shape))
print("Model output (logits) shape:", tuple(logits.shape))
print("\nModel output (logits) for the first 5 samples:")
print(logits[:5].cpu())

print("\nSoftmax probabilities for the first 5 samples:")
print(probs[:5].cpu())

print("\nPredicted class indices for the first 5 samples:")
print(pred_idx[:5].cpu().numpy())

print("\nClass mapping:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{i}: {class_name}")

Sample input shape: (64, 273)
Model output (logits) shape: (64, 3)

Model output (logits) for the first 5 samples:
tensor([[-3.6899, -4.6219,  4.7040],
        [-5.0255, -3.3542,  5.0909],
        [-7.2521,  6.9878, -2.2349],
        [-5.0313, -5.9272,  6.2137],
        [-3.8875, -4.1869,  4.5968]])

Softmax probabilities for the first 5 samples:
tensor([[2.2616e-04, 8.9054e-05, 9.9968e-01],
        [4.0401e-05, 2.1488e-04, 9.9974e-01],
        [6.5409e-07, 9.9990e-01, 9.8767e-05],
        [1.3073e-05, 5.3368e-06, 9.9998e-01],
        [2.0663e-04, 1.5317e-04, 9.9964e-01]])

Predicted class indices for the first 5 samples:
[2 2 1 2 2]

Class mapping:
0: Installment Payment
1: Loyalty Discount
2: No-Claim Protection Benefit


In [48]:
# Predict one example by passing feature values as function parameters
def predict_single_offer(**kwargs):
    required_features = features

    # Validate required inputs
    missing = [col for col in required_features if col not in kwargs]
    if missing:
        raise ValueError(f"Missing required features: {missing}")

    # Build one-row DataFrame in the same order as training
    one_sample = pd.DataFrame([{col: kwargs[col] for col in required_features}], columns=required_features)

    # Preprocess the single example
    processed = preprocessor.transform(one_sample)
    x_sample = torch.tensor(processed, dtype=torch.float32, device=device)

    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(x_sample)
        probs = torch.softmax(logits, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()

    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    prob_dict = {
        label_encoder.inverse_transform([i])[0]: float(probs[0, i].cpu())
        for i in range(len(label_encoder.classes_))
    }

    return {
        "predicted_class_index": pred_idx,
        "predicted_class": pred_label,
        "probabilities": prob_dict
    }


# Example: predict for a single customer
result = predict_single_offer(
    customer_age=35,
    customer_gender="Female",
    customer_occupation="Software Engineer",
    customer_tenure_years=4.5,
    policy_name="Term Life",
    premium_amount=12000,
    previous_premium_amount=11500,
    premium_increase_pct=4.35,
    payment_frequency="Annual",
    days_to_renewal=42,
    late_payment_count=1,
    avg_days_late=2.0,
    on_time_payment_rate=80.0,
    num_claims_last_year=0,
    total_claim_amount_last_year=0,
    rejected_claims=0,
    claims_approved=0
)

print(result)

{'predicted_class_index': 2, 'predicted_class': 'No-Claim Protection Benefit', 'probabilities': {'Installment Payment': 0.001418472034856677, 'Loyalty Discount': 0.001574478461407125, 'No-Claim Protection Benefit': 0.9970070719718933}}
